# 10 · From frozen models to competition predictions

**Question:** Can the selected recommender produce every required prediction, with a traceable model and data history?

The default mode runs the real native models on eight compact, verified competition sessions. Its output must match the corresponding rows from the complete run exactly. The full mode invokes the same resumable inference pipeline on every competition session. No hidden test labels or leaderboard score are available.

**Full submission already created:** `submission.csv.gz` contains all 5,015,409 task rows. The default run below creates only a 24-row review example. Use the download and submission section below for the complete file.


In [1]:
import hashlib
import json
import os
import subprocess
import time
from datetime import UTC, datetime
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display

STARTED = time.perf_counter()
FULL = os.environ.get("OTTO_FULL_INFERENCE") == "1"
location = Path.cwd().resolve()
ROOT = next(p for p in (location, *location.parents)
            if (p / "pyproject.toml").is_file()
            or ((p / "reports").is_dir() and (p / "configs").is_dir()))
print(datetime.now(UTC).isoformat(), "mode=", "full competition inference" if FULL else "verified native-model replay")


2026-09-09T01:54:42.209826+00:00 mode= verified native-model replay


## Model and data contract

Model weights are fixed by chronological model selection and evaluated before competition inference. Retrieval and historical statistics can then be refreshed from the official training events that precede the competition inputs. This refresh is a deployment operation; its outputs do not enter the reported temporal evaluation.

Set `OTTO_FULL_INFERENCE=1` in the managed execution environment to generate the complete prediction file. The managed run executes this mode with verified inputs, the locked project interpreter, and durable part checkpoints. Default replay requires no AWS access or training data.


In [2]:
if FULL:
    command = [str(ROOT / ".venv/bin/python"), str(ROOT / "scripts/run_inference.py"),
               "--stage", "predict", "--models", str(ROOT / "artifacts/research"),
               "--test", str(ROOT / "artifacts/test"), "--output", str(ROOT / "artifacts/inference"),
               "--threads", os.environ.get("OTTO_PREDICTION_THREADS", "4"),
               "--workers", os.environ.get("OTTO_PREDICTION_WORKERS", "1")]
    checkpoint_uri = os.environ.get("OTTO_PREDICTION_CHECKPOINT_URI")
    if checkpoint_uri:
        command += ["--checkpoint-uri", checkpoint_uri, "--owner-account", os.environ["OTTO_OWNER_ACCOUNT"],
                    "--region", os.environ["OTTO_AWS_REGION"]]
    completed = subprocess.run(command, cwd=ROOT, check=True, capture_output=True, text=True)
    print(completed.stdout)
    full = json.loads((ROOT / "artifacts/inference/prediction/manifest.json").read_text())
    destination = ROOT / "artifacts/inference/prediction/submission.csv.gz"
    preview = pd.read_csv(destination, nrows=9)
    assert full["status"] == "passed"
else:
    bundle = ROOT / "reports/research/inference_replay"
    manifest = json.loads((bundle / "manifest.json").read_text())
    assert manifest["status"] == "passed"
    for name, expected in manifest["files"].items():
        assert hashlib.sha256((bundle / name).read_bytes()).hexdigest() == expected, name
    candidates = pd.read_parquet(bundle / "features.parquet")
    records = []
    objectives = ("clicks", "carts", "orders")
    models = {o: lgb.Booster(model_file=str(bundle / manifest["models"][o]["path"])) for o in objectives}
    for session, query in candidates.groupby("session", sort=True):
        aids = query["aid"].to_numpy()
        for objective in objectives:
            names = manifest["models"][objective]["features"]
            assert models[objective].feature_name() == names
            score = models[objective].predict(query[names].to_numpy(dtype=np.float32), num_threads=1)
            order = np.lexsort((aids, -score))[:20]
            records.append((f"{session}_{objective}", " ".join(map(str, aids[order]))))
    replay = pd.DataFrame(records, columns=["session_type", "labels"])
    destination = ROOT / "artifacts/inference_replay.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    replay.to_csv(destination, index=False)
    assert destination.read_bytes() == (bundle / "expected.csv").read_bytes()
    full = manifest["full_prediction"]
    preview = replay.head(9)
    print(f"Exact replay passed: {len(replay):,} rows, {manifest['sessions']} real competition sessions")


Exact replay passed: 24 rows, 8 real competition sessions


## Coverage and output validation

Every observed session must appear exactly once for each of clicks, carts, and orders. Each recommendation list contains 20 unique nonnegative item IDs. The full validator checks the exact session ledger, objective coverage, duplicate rows, list lengths, and the final file checksum. Partial or incompatible artifacts cannot certify completion.


In [3]:
display(pd.DataFrame([
    ("Complete competition sessions", f"{full['sessions']:,}"),
    ("Required task rows", f"{full['rows']:,}"),
    ("Prediction artifact SHA-256", full["sha256"]),
    ("Prediction input identity", full["input_id"]),
], columns=["Verified evidence", "Value"]))
assert full["rows"] == 3 * full["sessions"]
for labels in preview["labels"]:
    items = labels.split()
    assert len(items) == len(set(items)) == 20
    assert all(item.isdecimal() for item in items)
display(preview)


,Verified evidence,Value
0,Complete competition sessions,"1,671,803"
1,Required task rows,"5,015,409"
2,Prediction artifact SHA-256,adc1c7d496249b8a37550c4c077dba8d12bc413d0fe89a...
3,Prediction input identity,2b001d27dedaa6b684675cfc0facb65dc5c2ef85b07758...


,session_type,labels
0,12899779_clicks,59625 875854 737445 1708367 1727247 943394 180...
1,12899779_carts,875854 59625 731692 260336 834466 1727247 6579...
2,12899779_orders,875854 59625 731692 260336 1340695 941596 1157...
3,12899780_clicks,1142000 260305 736515 582732 973453 260138 119...
4,12899780_carts,260305 1142000 973453 582732 736515 1088246 16...
5,12899780_orders,260305 1142000 582732 736515 973453 1639095 21...
6,12899781_clicks,918667 199008 1714342 644935 91240 273625 8202...
7,12899781_carts,918667 199008 428697 1714342 205149 811448 820...
8,12899781_orders,918667 199008 205149 428697 57315 643158 14173...


## Download and submit the completed full run

**The code has already generated the full competition file. Kaggle upload is a separate action.**

A submission is a CSV of recommended product IDs. It has the two columns `session_type`
and `labels`: one row per session and action (clicks, carts, orders), with 20
space-separated product IDs in each recommendation list.

| File | Contents | Use |
|---|---|---|
| `submission.csv.gz` | 5,015,409 prediction rows for all 1,671,803 test sessions | Full competition submission |
| `inference_replay.csv` | 24 rows for eight example sessions | Review example; do not upload to Kaggle |

The full file was produced by this notebook in managed full mode on September 8, 2026.
It is **296,087,864 bytes** (about 296 MB) and is stored durably in the project S3 bucket.
Its SHA-256 is `adc1c7d496249b8a37550c4c077dba8d12bc413d0fe89a80221472cd813a7ef3`.

1. [Open the completed full file in AWS](https://s3.console.aws.amazon.com/s3/object/otto-recsys-560403859723-us-west-2?region=us-west-2&bucketType=general&prefix=ranking%2Fresearch%2F55ad451e895863af311e4a917a6fe0d4ab9165ad6b406fffb066d25bf0af4754%2Fdelivery%2Ff9c2c07590a24e66d611b09d2c77352d690a9a0ea663155afce184b7fcd98d37%2Finference%2Fprediction%2Fsubmission.csv.gz), sign in to the project AWS account if needed, and choose **Download** for `submission.csv.gz`.
2. Sign in to [the OTTO competition on Kaggle](https://www.kaggle.com/competitions/otto-recommender-system) and choose **Late Submission**. The competition's original deadline was January 31, 2023. Its public page displays the late-submission control; this control is disabled while signed out, so account-specific availability must be checked after sign-in.
3. Select the full `submission.csv.gz` file and complete Kaggle's upload. Check Kaggle's processing result before recording any score. If the upload form requests an uncompressed CSV, extract the gzip file to `submission.csv`; renaming the extension does not decompress it.

No retraining or full inference rerun is needed to use this already completed output.
The local **0.584392** validation score is not a Kaggle submission score.
The notebook does not log into Kaggle or submit automatically.


## Decision and limits

A prediction file is an engineering deliverable, not a measured hidden-test score. The portfolio's quality claims come from the controlled temporal evaluation in notebook 09. The complete competition output is generated and validated in full mode; the default notebook proves a small exact native-model replay for convenient review. There is no claim of Kaggle acceptance or a leaderboard position.

The model is frozen during both modes. Rerunning an identical full job checks completed part digests and resumes missing work. A changed test input, retrieval graph, model seal, or feature implementation requires a distinct experiment workspace.


In [4]:
print(datetime.now(UTC).isoformat(), "inference_notebook_complete",
      f"elapsed_seconds={time.perf_counter()-STARTED:.3f}")
print("Full competition submission:" if FULL else "Review example only (24 rows; do not upload):", destination)
print("Kaggle upload is a separate action; this notebook does not submit automatically.")
print("Runtime:", {"lightgbm": lgb.__version__, "numpy": np.__version__, "pandas": pd.__version__})


2026-09-09T01:54:42.351936+00:00 inference_notebook_complete elapsed_seconds=0.142
Review example only (24 rows; do not upload): /tmp/otto-notebooks-h881m2pe/artifacts/inference_replay.csv
Kaggle upload is a separate action; this notebook does not submit automatically.
Runtime: {'lightgbm': '4.7.0', 'numpy': '2.5.3', 'pandas': '3.0.5'}
